In [1]:
# Set your username here - use it consistently across all resources
USERNAME = "petboga"

In [2]:
import datetime
import json

import boto3
import requests

In [13]:
# Try different dates to see how the data changes
DATE_PARAM = "2025-11-14"

date = datetime.datetime.strptime(DATE_PARAM, "%Y-%m-%d")

# Construct the API URL
url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/{date.strftime('%Y/%m/%d')}"
print(f"Requesting REST API URL: {url}")

# Make the API request
wiki_server_response = requests.get(url, headers={"User-Agent": "curl/7.68.0"})
wiki_response_status = wiki_server_response.status_code
wiki_response_body = wiki_server_response.text

print(f"Wikipedia REST API Response body: {wiki_response_body[:500]}...")
print(f"Wikipedia REST API Response Code: {wiki_response_status}")

# Validate response
if wiki_response_status != 200:
    raise Exception(f"Received non-OK status code from Wiki Server: {wiki_response_status}")
print(f"Successfully retrieved Wikipedia data, content-length: {len(wiki_response_body)}")

Requesting REST API URL: https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/2025/11/14
Wikipedia REST API Response body: {"items":[{"project":"en.wikipedia","access":"all-access","year":"2025","month":"11","day":"14","articles":[{"article":"Main_Page","views":5841654,"rank":1},{"article":"Special:Search","views":872268,"rank":2},{"article":"2025_Bihar_Legislative_Assembly_election","views":823147,"rank":3},{"article":"Google_Chrome","views":300714,"rank":4},{"article":"Election_Commission_of_India","views":260337,"rank":5},{"article":"Wikipedia:Featured_pictures","views":252437,"rank":6},{"article":"Bihar_Legislat...
Wikipedia REST API Response Code: 200
Successfully retrieved Wikipedia data, content-length: 57742


In [14]:
# Parse the API response and extract top edits
wiki_response_parsed = wiki_server_response.json()
most_viewed = wiki_response_parsed["items"][0]["articles"]
# top_edits = wiki_response_parsed["items"][0]["results"][0]["top"]

# Transform to JSON Lines format
current_time = datetime.datetime.now(datetime.timezone.utc)
json_lines = ""
# for page in most_viewed[:5]:
for page in most_viewed:    
    record = {
        "title": page["article"],
        "views": page["views"],
        "rank": page["rank"],
        "date": date.strftime("%Y-%m-%d"),
        "retrieved_at": current_time.replace(tzinfo=None).isoformat(),
    }
    json_lines += json.dumps(record) + "\n"

print(f"Transformed {len(most_viewed)} records to JSON Lines")
print(f"First few lines:\n{json_lines[:500]}...")

print(len(most_viewed))

Transformed 1000 records to JSON Lines
First few lines:
{"title": "Main_Page", "views": 5841654, "rank": 1, "date": "2025-11-14", "retrieved_at": "2025-12-11T16:07:10.773773"}
{"title": "Special:Search", "views": 872268, "rank": 2, "date": "2025-11-14", "retrieved_at": "2025-12-11T16:07:10.773773"}
{"title": "2025_Bihar_Legislative_Assembly_election", "views": 823147, "rank": 3, "date": "2025-11-14", "retrieved_at": "2025-12-11T16:07:10.773773"}
{"title": "Google_Chrome", "views": 300714, "rank": 4, "date": "2025-11-14", "retrieved_at": "2025-12-11T1...
1000


In [7]:
S3_WIKI_BUCKET = "petboga-wikidata"
s3 = boto3.client("s3")

# Check if the bucket exists
bucket_names = [bucket["Name"] for bucket in s3.list_buckets()["Buckets"]]
if S3_WIKI_BUCKET not in bucket_names:
    # LAB 1: Create the bucket if it doesn't exist
    # YOUR SOLUTION COMES HERE =========================
    response = s3.create_bucket(
    Bucket=S3_WIKI_BUCKET,
    CreateBucketConfiguration={
        'LocationConstraint': 'eu-west-1',
    },
)
    # ==================================================
    print(f"Created new bucket: {S3_WIKI_BUCKET}")
else:
    print(f"Using existing bucket: {S3_WIKI_BUCKET}")


Using existing bucket: petboga-wikidata


In [8]:
# Test Lab 1
assert USERNAME != "<username>", "Please set your USERNAME at the top of the notebook"
assert S3_WIKI_BUCKET.endswith("-wikidata"), "Bucket name must end with '-wikidata'"

try:
    s3.head_bucket(Bucket=S3_WIKI_BUCKET)
    print(f"Bucket {S3_WIKI_BUCKET} exists!")
except Exception as e:
    print(f"Bucket {S3_WIKI_BUCKET} not found: {e}")
    raise

Bucket petboga-wikidata exists!


In [15]:

# raw-views/raw-views-YYYY-MM-DD.json
response = s3.put_object(
    Body=json_lines,
    Bucket=S3_WIKI_BUCKET,
    Key='raw-views/raw-views-'+datetime.datetime.strftime(date,"%Y-%m-%d")+'.json'
)

print(response)

{'ResponseMetadata': {'RequestId': 'H8DVJ952ASAZJKBA', 'HostId': 'KrEylubeH+ECM4+6LtUdpOywFYQFtfwIRxRURvH8stqEbUormAccCuK9ip336HG9mqk4U3sKQosWXn8ZuOeIxfKT1wSlhLdv', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'KrEylubeH+ECM4+6LtUdpOywFYQFtfwIRxRURvH8stqEbUormAccCuK9ip336HG9mqk4U3sKQosWXn8ZuOeIxfKT1wSlhLdv', 'x-amz-request-id': 'H8DVJ952ASAZJKBA', 'date': 'Thu, 11 Dec 2025 16:07:21 GMT', 'x-amz-server-side-encryption': 'AES256', 'etag': '"e6c98e5f8d5e546bfdad89f24d37776a"', 'x-amz-checksum-crc32': 'SeiRqg==', 'x-amz-checksum-type': 'FULL_OBJECT', 'content-length': '0', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'ETag': '"e6c98e5f8d5e546bfdad89f24d37776a"', 'ChecksumCRC32': 'SeiRqg==', 'ChecksumType': 'FULL_OBJECT', 'ServerSideEncryption': 'AES256'}
